# TookaBERT rating-band training

Upload two things to Google Drive before running this notebook.

1. The folder `experiments/baselines/tookabert-epoch2-rating-band` from this project. It is the epoch-2 encoder with a new untrained head for `low`, `medium`, and `high`. Put it at `MyDrive/digikala-experiments/rating-band/model` so that `model.safetensors`, `config.json`, `tokenizer.json`, and `tokenizer_config.json` sit directly in that folder.
2. A CSV of the preprocessed reviews, with `body` and either `rating_band` or `rate`. Put it at `MyDrive/digikala-experiments/rating-band/reviews.csv`.

Bands are `low` for 0 ≤ rate < 2.5, `medium` for 2.5 ≤ rate < 4, and `high` for 4 ≤ rate ≤ 5. Rows outside that range are dropped. Each body then gets the epoch-2 letter, digit, and emoji stages. The text is not stemmed again.

The split is 70% train, 15% validation, 15% test, stratified by `rating_band`, seed 2026. Training unfreezes blocks 8–11 and the new head for one epoch. The loss is class-weighted cross-entropy. Checkpoints and metrics go to Drive. There is no MLflow step.

`BATCH_SIZE` is 64 and evaluation uses 128. On the earlier Colab runs, batch 16 used about 3–4 GB of a 15 GB GPU. The lower eight blocks stay frozen, so 64 is the size aimed at that card. If the runtime stops with an out-of-memory error, set `BATCH_SIZE` to 32 and run the training cell again. A finished epoch is not trained again.


In [ ]:
!pip -q install polars


In [ ]:
MODEL_NAME = "PartAI/TookaBERT-Base"
DRIVE_DIR = "/content/drive/MyDrive/digikala-experiments/rating-band"
MODEL_DIR = DRIVE_DIR + "/rating-band-model"
CSV_PATH = DRIVE_DIR + "/rating-band-reviews.csv"

MAX_LENGTH = 128
EPOCHS = 1
BATCH_SIZE = 64
EVAL_BATCH_SIZE = 128
GRAD_ACCUM = 1
BLOCK_LR = 2e-5
HEAD_LR = 1e-4
TOP_BLOCKS = 4
SEED = 2026
SPLIT_SEED = 2026
TRAIN_SHARE = 0.70
VALIDATION_SHARE = 0.15
SAVE_STEPS = 1000

LABELS = ["low", "medium", "high"]
LABEL_CODE = {name: index for index, name in enumerate(LABELS)}


In [ ]:
from pathlib import Path

import torch
from google.colab import drive

drive.mount("/content/drive")
model_dir = Path(MODEL_DIR)
csv_path = Path(CSV_PATH)
needed = ["model.safetensors", "config.json", "tokenizer.json", "tokenizer_config.json"]
missing = [name for name in needed if not (model_dir / name).is_file()]
if missing:
    raise FileNotFoundError("Upload the rating-band model to " + str(model_dir) + ". Missing: " + ", ".join(missing))
if not csv_path.is_file():
    raise FileNotFoundError("Upload the reviews CSV to " + str(csv_path))
Path(DRIVE_DIR, "runs").mkdir(parents=True, exist_ok=True)
Path(DRIVE_DIR, "checkpoints").mkdir(parents=True, exist_ok=True)
Path(DRIVE_DIR, "logs").mkdir(parents=True, exist_ok=True)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(props.name, "memory_gb", round(props.total_memory / 1024**3, 1))
else:
    print("cpu")
print("model", model_dir)
print("csv", csv_path, csv_path.stat().st_size)


Mounted at /content/drive
Tesla T4 memory_gb 14.6
model /content/drive/MyDrive/digikala-experiments/rating-band/rating-band-model
csv /content/drive/MyDrive/digikala-experiments/rating-band/rating-band-reviews.csv 625709515


## Text and bands

The prepared table is saved on Drive. A rerun loads it instead of rewriting every body.


In [ ]:
"""Character rules for the 12 November 2026 preprocessing experiment.

The input is the already cleaned, already Hazm-stemmed comments file.
This pass does not stem again, does not drop Latin letters, and does not
remove Persian stopwords. Stages run in order: letters, then digits,
then emoji.
"""

from __future__ import annotations

import re
import unicodedata

_ARABIC_LETTERS = str.maketrans(
    {
        "ك": "ک",
        "ي": "ی",
        "ى": "ی",
        "أ": "ا",
        "إ": "ا",
        "ٱ": "ا",
        "ة": "ه",
        "ۀ": "ه",
        "ھ": "ه",
        "ؤ": "و",
    }
)

_DIGITS = str.maketrans(
    {chr(ord("0") + i): chr(0x06F0 + i) for i in range(10)}
    | {chr(0x0660 + i): chr(0x06F0 + i) for i in range(10)}
)

_DIACRITICS = re.compile(r"[\u064B-\u065F\u0670\u06D6-\u06ED\u0640]")
_INVISIBLE = re.compile(r"[\u200B\u200D\uFEFF\u200E\u200F\u202A-\u202E\u2066-\u2069]")
_REPEATED_ZWNJ = re.compile(r"\u200C{2,}")
_EDGE_ZWNJ = re.compile(r"(?<!\S)\u200C|\u200C(?!\S)")
_URL = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
_EMAIL = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
_CHAR_RUN = re.compile(r"(.)\1{2,}")
_LINE_BREAKS = re.compile(r"[\r\n\t]+")
_SPACES = re.compile(r"\s+")
_EMOJI = re.compile(
    "["
    "\U0001F1E6-\U0001F1FF"
    "\U0001F300-\U0001FAFF"
    "\U00002600-\U000027BF"
    "\U0000FE0E\U0000FE0F"
    "]+"
)
_PURE_PERSIAN_DIGITS = re.compile(r"^[۰-۹]+$")


def _squash(text: str) -> str:
    text = _LINE_BREAKS.sub(" ", text)
    text = _SPACES.sub(" ", text).strip()
    return _EDGE_ZWNJ.sub("", text)


def apply_letters(text: str) -> str:
    """NFKC, Persian letters, one interior half-space, no URLs, no long runs.

    Digits, emoji, and Latin letters are left for later stages. Nothing
    is stemmed.
    """
    if not isinstance(text, str) or not text:
        return ""
    out = unicodedata.normalize("NFKC", text)
    out = out.translate(_ARABIC_LETTERS)
    out = _DIACRITICS.sub("", out)
    out = _INVISIBLE.sub("", out)
    out = _REPEATED_ZWNJ.sub("\u200C", out)
    out = _EDGE_ZWNJ.sub("", out)
    out = _URL.sub(" ", out)
    out = _EMAIL.sub(" ", out)
    out = _CHAR_RUN.sub(r"\1\1", out)
    return _squash(out)


def apply_digits(text: str) -> str:
    """Fold ASCII and Arabic-Indic digits to Persian, then drop pure numbers.

    A token that mixes digits and letters, such as ``A54``, stays.
    """
    if not isinstance(text, str) or not text:
        return ""
    folded = text.translate(_DIGITS)
    kept = [token for token in folded.split(" ") if token and _PURE_PERSIAN_DIGITS.fullmatch(token) is None]
    return " ".join(kept)


def apply_emoji(text: str) -> str:
    """Delete emoji characters. The caller drops a body that becomes empty."""
    if not isinstance(text, str) or not text:
        return ""
    return _squash(_EMOJI.sub(" ", text))


def prepare_body(text):
    return apply_emoji(apply_digits(apply_letters(text or "")))

import time
import polars as pl

prepared_path = Path(DRIVE_DIR) / "prepared.parquet"
if prepared_path.is_file():
    prepared = pl.read_parquet(prepared_path)
    print("prepared already saved", prepared.height)
else:
    frame = pl.read_csv(csv_path, infer_schema_length=10_000)
    first = frame.columns[0]
    if first.startswith("\ufeff"):
        frame = frame.rename({first: first.lstrip("\ufeff")})
    if "rating_band" in frame.columns:
        frame = frame.select(
            pl.col("body").cast(pl.String).fill_null(""),
            pl.col("rating_band").cast(pl.String),
        )
    else:
        rate = pl.col("rate").cast(pl.Float64, strict=False)
        band = (
            pl.when((rate >= 0) & (rate < 2.5)).then(pl.lit("low"))
            .when((rate >= 2.5) & (rate < 4)).then(pl.lit("medium"))
            .when((rate >= 4) & (rate <= 5)).then(pl.lit("high"))
            .otherwise(pl.lit("invalid"))
        )
        frame = frame.select(pl.col("body").cast(pl.String).fill_null(""), band.alias("rating_band"))
    frame = frame.filter(pl.col("rating_band").is_in(LABELS) & (pl.col("body") != ""))
    bodies = frame.get_column("body").to_list()
    rewritten = []
    started = time.perf_counter()
    total = len(bodies)
    for index, body in enumerate(bodies, start=1):
        rewritten.append(prepare_body(body))
        if index == total or index % 100_000 == 0:
            elapsed = max(time.perf_counter() - started, 0.001)
            print(f"text {index} of {total} ({index / elapsed:.0f} rows/s)", flush=True)
    prepared = frame.with_columns(pl.Series("body", rewritten)).filter(pl.col("body") != "")
    prepared = prepared.with_columns(pl.col("rating_band").replace_strict(LABEL_CODE).alias("label_id"))
    prepared.write_parquet(prepared_path)
    print("prepared", prepared.height)
print(prepared.group_by("rating_band").len().sort("rating_band"))


text 100000 of 5258342 (14826 rows/s)
text 200000 of 5258342 (16713 rows/s)
text 300000 of 5258342 (18146 rows/s)
text 400000 of 5258342 (17331 rows/s)
text 500000 of 5258342 (18193 rows/s)
text 600000 of 5258342 (18783 rows/s)
text 700000 of 5258342 (18205 rows/s)
text 800000 of 5258342 (18665 rows/s)
text 900000 of 5258342 (18797 rows/s)
text 1000000 of 5258342 (18545 rows/s)
text 1100000 of 5258342 (18827 rows/s)
text 1200000 of 5258342 (18822 rows/s)
text 1300000 of 5258342 (18786 rows/s)
text 1400000 of 5258342 (19011 rows/s)
text 1500000 of 5258342 (18733 rows/s)
text 1600000 of 5258342 (18911 rows/s)
text 1700000 of 5258342 (19084 rows/s)
text 1800000 of 5258342 (18887 rows/s)
text 1900000 of 5258342 (19006 rows/s)
text 2000000 of 5258342 (19118 rows/s)
text 2100000 of 5258342 (18897 rows/s)
text 2200000 of 5258342 (19045 rows/s)
text 2300000 of 5258342 (19012 rows/s)
text 2400000 of 5258342 (18981 rows/s)
text 2500000 of 5258342 (19124 rows/s)
text 2600000 of 5258342 (19013 row

## Train

Validation and test stay human-labeled bands. The new head and blocks 8–11 are the trainable parameters. Run the cell again after an interrupt to resume from the last Drive checkpoint.


In [6]:
import inspect
import json

import numpy as np
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

parts = []
for label in LABELS:
    group = prepared.filter(pl.col("rating_band") == label)
    shuffled = group.sample(fraction=1.0, shuffle=True, seed=SPLIT_SEED)
    count = shuffled.height
    n_train = int(count * TRAIN_SHARE)
    n_validation = int(count * VALIDATION_SHARE)
    n_test = count - n_train - n_validation
    parts.append(shuffled.with_columns(
        pl.Series("split", (["train"] * n_train) + (["validation"] * n_validation) + (["test"] * n_test))
    ))
gold = pl.concat(parts)
train_frame = gold.filter(pl.col("split") == "train")
valid_frame = gold.filter(pl.col("split") == "validation")
test_frame = gold.filter(pl.col("split") == "test")
print("train", train_frame.height, "validation", valid_frame.height, "test", test_frame.height)

run_dir = Path(DRIVE_DIR) / "runs" / "PartAI--TookaBERT-Base--rating-band"
checkpoint_root = Path(DRIVE_DIR) / "checkpoints" / "PartAI--TookaBERT-Base-rating-band"
run_dir.mkdir(parents=True, exist_ok=True)
checkpoint_root.mkdir(parents=True, exist_ok=True)


class ReviewSet(Dataset):
    def __init__(self, frame):
        self.texts = frame.get_column("body").to_list()
        self.labels = frame.get_column("label_id").to_list()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        encoded = tokenizer(self.texts[index] or "", truncation=True, max_length=MAX_LENGTH)
        encoded["labels"] = int(self.labels[index])
        return encoded


def metric_bundle(actual, predicted):
    matrix = [[0, 0, 0] for _ in LABELS]
    for truth, guess in zip(actual, predicted):
        matrix[int(truth)][int(guess)] += 1
    report = {"confusion_matrix": matrix}
    f1s = []
    correct = 0
    total = 0
    for index, name in enumerate(LABELS):
        support = sum(matrix[index])
        predicted_count = sum(matrix[row][index] for row in range(3))
        true_positive = matrix[index][index]
        precision = true_positive / predicted_count if predicted_count else 0.0
        recall = true_positive / support if support else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if precision + recall else 0.0
        f1s.append(f1)
        correct += true_positive
        total += support
        report[f"{name}_precision"] = float(precision)
        report[f"{name}_recall"] = float(recall)
        report[f"{name}_f1"] = float(f1)
        report[f"{name}_support"] = int(support)
    report["accuracy"] = correct / total if total else 0.0
    report["f1_macro"] = float(sum(f1s) / len(f1s))
    report["f1_micro"] = report["accuracy"]
    report["rows"] = int(total)
    return report


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    scores = metric_bundle(labels, np.argmax(logits, axis=-1))
    return {"accuracy": scores["accuracy"], "f1": scores["f1_macro"]}


counts = np.bincount(np.array(train_frame.get_column("label_id").to_list()), minlength=3).astype(np.float64)
class_weights = torch.tensor(len(train_frame) / (3.0 * counts), dtype=torch.float)
print("class weights", [round(float(value), 4) for value in class_weights])

tokenizer = AutoTokenizer.from_pretrained(model_dir)
shared = dict(
    output_dir=str(checkpoint_root),
    learning_rate=BLOCK_LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=4,
    dataloader_pin_memory=torch.cuda.is_available(),
)
try:
    args = TrainingArguments(eval_strategy="epoch", **shared)
except TypeError:
    args = TrainingArguments(evaluation_strategy="epoch", **shared)
token_kwargs = (
    {"processing_class": tokenizer}
    if "processing_class" in inspect.signature(Trainer.__init__).parameters
    else {"tokenizer": tokenizer}
)


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(
            outputs.logits, labels, weight=class_weights.to(outputs.logits.device)
        )
        return (loss, outputs) if return_outputs else loss


def score_frame(model, frame):
    score_args = TrainingArguments(
        output_dir="/tmp/score-rating-band",
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        report_to="none",
        fp16=torch.cuda.is_available(),
        dataloader_pin_memory=torch.cuda.is_available(),
    )
    scorer = Trainer(
        model=model,
        args=score_args,
        data_collator=DataCollatorWithPadding(tokenizer),
        **token_kwargs,
    )
    prediction = scorer.predict(ReviewSet(frame))
    return metric_bundle(prediction.label_ids, np.argmax(prediction.predictions, axis=-1))


def write_json(name, payload):
    payload = {"model_name": MODEL_NAME, "stage": "rating-band", "started_from": str(model_dir), **payload}
    (run_dir / name).write_text(json.dumps(payload, indent=2))
    log_path = Path(DRIVE_DIR) / "logs" / "PartAI--TookaBERT-Base--rating-band.jsonl"
    with log_path.open("a") as handle:
        handle.write(json.dumps(payload) + "\n")
    shown = {
        key: round(payload[key], 4)
        for key in ("accuracy", "f1_macro", "f1_micro", "train_loss")
        if key in payload
    }
    print(payload.get("split", name), shown)


weights_ready = (run_dir / "model.safetensors").is_file()
done = weights_ready and (run_dir / "metrics.json").is_file() and (run_dir / "test_metrics.json").is_file()
if done:
    print("training already saved", run_dir)
else:
    model = None
    if not weights_ready:
        model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        for name, param in model.named_parameters():
            param.requires_grad = not name.startswith("bert.")
        for layer in model.bert.encoder.layer[-TOP_BLOCKS:]:
            for param in layer.parameters():
                param.requires_grad = True
        trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)
        print("trainable", trainable)
        head, block = [], []
        for name, param in model.named_parameters():
            if param.requires_grad:
                (block if name.startswith("bert.") else head).append(param)
        trainer = WeightedTrainer(
            model=model,
            args=args,
            train_dataset=ReviewSet(train_frame),
            eval_dataset=ReviewSet(valid_frame),
            data_collator=DataCollatorWithPadding(tokenizer),
            compute_metrics=compute_metrics,
            **token_kwargs,
        )

        def create_optimizer():
            trainer.optimizer = torch.optim.AdamW(
                [{"params": block, "lr": BLOCK_LR}, {"params": head, "lr": HEAD_LR}],
                weight_decay=0.01,
            )
            return trainer.optimizer

        trainer.create_optimizer = create_optimizer
        last = get_last_checkpoint(str(checkpoint_root))
        print("resume checkpoint", last)
        trainer.train(resume_from_checkpoint=last)
        model.save_pretrained(run_dir)
        tokenizer.save_pretrained(run_dir)
        print("saved weights before scoring", run_dir)
        if torch.cuda.is_available():
            print("peak_memory_gb", round(torch.cuda.max_memory_allocated() / 1024**3, 2))
        train_row = None
        for row in reversed(trainer.state.log_history):
            if "train_loss" in row or "loss" in row:
                train_row = row
                break
        write_json("train_metrics.json", {"split": "train", "log": train_row})
    if model is None:
        model = AutoModelForSequenceClassification.from_pretrained(run_dir)
    if not (run_dir / "metrics.json").is_file():
        write_json("metrics.json", {"split": "validation", **score_frame(model, valid_frame)})
    if not (run_dir / "test_metrics.json").is_file():
        write_json("test_metrics.json", {"split": "test", **score_frame(model, test_frame)})


train 3680692 validation 788719 test 788724
class weights [1.7687, 1.8503, 0.5279]


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable 28353795
resume checkpoint None


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:437: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.882636,0.821390,0.658971,0.583642


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (58 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved weights before scoring /content/drive/MyDrive/digikala-experiments/rating-band/runs/PartAI--TookaBERT-Base--rating-band
peak_memory_gb 1.86
train {}


validation {'accuracy': 0.659, 'f1_macro': 0.5836, 'f1_micro': 0.659}


test {'accuracy': 0.6577, 'f1_macro': 0.5822, 'f1_micro': 0.6577}
